# RAG Chatbot Lich Su Viet Nam

He thong hoi dap thong minh ve **Lich Su Viet Nam** su dung RAG.

### Pipeline
1. Thiet lap moi truong
2. Doc & xu ly PDF
3. Phan doan van ban (Chunking)
4. Nhung vector (Embedding)
5. Co so du lieu vector (FAISS + Chroma)
6. Benchmark
7. Hoi dap da phuong thuc
8. Agent thong minh (LangGraph)
9. Fine-tuning (LoRA)
10. Danh gia toan dien
11. API + Giao dien web

In [1]:
# Cai dat thu vien
!pip install -q pymupdf sentence-transformers faiss-cpu chromadb python-dotenv \
    groq langchain langchain-community langchain-groq langgraph \
    duckduckgo-search pandas matplotlib tqdm peft transformers datasets trl accelerate \
    fastapi uvicorn httpx pytesseract Pillow pdf2image gTTS numpy

In [2]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, re, time, pickle, math
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# === CAU HINH ===
CFG = {
    "pdf_dir":      Path("data/pdf"),
    "text_dir":     Path("data/extracted_text"),
    "chunk_dir":    Path("data/chunks"),
    "emb_dir":      Path("data/embeddings"),
    "vs_faiss":     Path("vector_stores/faiss"),
    "vs_chroma":    Path("vector_stores/chroma"),
    "out_dir":      Path("outputs"),
    "dataset_dir":  Path("data/dataset"),
    "embed_model":  "paraphrase-multilingual-MiniLM-L12-v2",
    "llm_model":    "llama-3.3-70b-versatile",
    "collection":   "lich_su_vn",
}

# === GROQ CLIENT ===
from groq import Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def hoi_llm(prompt, max_tokens=2048):
    """Goi Groq LLM de sinh van ban."""
    resp = groq_client.chat.completions.create(
        model=CFG["llm_model"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

print("Cau hinh OK — Groq +", CFG["llm_model"])

Cau hinh OK — Groq + llama-3.3-70b-versatile


---
## Phan 1: Thiet lap moi truong

In [4]:
def khoi_tao():
    thu_muc = [
        CFG["pdf_dir"], CFG["text_dir"], CFG["chunk_dir"], CFG["emb_dir"],
        CFG["vs_faiss"], CFG["vs_chroma"], CFG["dataset_dir"],
        CFG["out_dir"] / "benchmark", CFG["out_dir"] / "evaluation",
        Path("data/test_image"),
    ]
    tao_moi = 0
    for d in thu_muc:
        if not d.exists():
            d.mkdir(parents=True, exist_ok=True)
            tao_moi += 1
    print(f"Thu muc: {len(thu_muc)} tong | {tao_moi} tao moi")

    api_key = os.getenv("GROQ_API_KEY", "")
    if not api_key:
        print("[!] GROQ_API_KEY chua co trong .env")
        return False
    print(f"API Key: {api_key[:8]}...{api_key[-4:]}")

    ds_pdf = sorted(CFG["pdf_dir"].glob("*.pdf"))
    if ds_pdf:
        for f in ds_pdf:
            mb = f.stat().st_size / 1024 / 1024
            print(f"  PDF: {f.name} ({mb:.1f} MB)")
    else:
        print("[!] Chua co PDF - dat file vao data/pdf/")
    return True

khoi_tao()

Thu muc: 10 tong | 0 tao moi
API Key: gsk_wNHy...WaUx
  PDF: Lịch sử Việt Nam tập 15 Từ năm 1986 đến năm 2000-Nguyễn Ngọc Mão-2017.pdf (103.3 MB)


True

---
## Phan 2: Doc & xu ly PDF
**PyMuPDF** doc truc tiep, fallback **pytesseract** OCR cho trang scan.

In [5]:
def doc_pdf(duong_dan, force=False):
    file_luu = CFG["text_dir"] / "noi_dung.txt"
    file_meta = CFG["text_dir"] / "thong_tin.json"

    if file_luu.exists() and not force:
        print(f"Da co: {file_luu}")
        return file_luu.read_text(encoding="utf-8")

    import fitz
    from tqdm import tqdm

    tai_lieu = fitz.open(str(duong_dan))
    so_trang = len(tai_lieu)
    print(f"Dang xu ly: {Path(duong_dan).name} ({so_trang} trang)")

    ds_trang = []
    for i in tqdm(range(so_trang), desc="Doc PDF"):
        trang = tai_lieu[i]
        noi_dung = trang.get_text().strip()
        if len(noi_dung) < 30:
            try:
                import pytesseract
                from PIL import Image
                import io
                anh = trang.get_pixmap(dpi=300)
                img = Image.open(io.BytesIO(anh.tobytes("png")))
                noi_dung = pytesseract.image_to_string(img, lang="vie").strip()
            except Exception:
                pass
        ds_trang.append({"trang": i + 1, "text": noi_dung, "so_ky_tu": len(noi_dung)})

    tai_lieu.close()

    van_ban_day_du = "\n\n".join(
        f"--- Trang {t['trang']}/{so_trang} ---\n{t['text']}" for t in ds_trang
    )

    file_luu.write_text(van_ban_day_du, encoding="utf-8")
    file_meta.write_text(json.dumps({
        "nguon": str(duong_dan), "so_trang": so_trang,
        "tong_ky_tu": len(van_ban_day_du), "chi_tiet": ds_trang
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    trang_it = sum(1 for t in ds_trang if t["so_ky_tu"] < 50)
    print(f"Ket qua: {so_trang} trang | {len(van_ban_day_du):,} ky tu")
    if trang_it:
        print(f"  Canh bao: {trang_it} trang it noi dung")
    return van_ban_day_du

In [6]:
ds_pdf = sorted(CFG["pdf_dir"].glob("*.pdf"))
if ds_pdf:
    van_ban = doc_pdf(ds_pdf[0])
    print(f"\nXem truoc:\n{'='*50}\n{van_ban[:600]}")
else:
    van_ban = ""
    print("Chua co PDF!")

Da co: data/extracted_text/noi_dung.txt

Xem truoc:
--- Trang 1/451 ---
VIỆN H À N  LÂM  K H O A  H ỌC  XẢ H ỘI VIỆT N A M  
VIỆN SỬ HỌC
NGUYỄN NGỌC M ÃO
(Chù biên)
LỊCH SỬ 
VIỆT NAM
TẬP 15 TỪ N Ă M  1986 Đ ẾN  N Ă M  2000

--- Trang 2/451 ---


--- Trang 3/451 ---
LỊCH SỬ VIỆT NAM
TẬP 15 
TÙ NĂM 1986 ĐÉN NĂM 2000

--- Trang 4/451 ---
Biên mục trên xuất bản phẩm của Thư viện Quốc gia Việt Nam
Nguyễn Ngọc Mão
Lịch sử Việt Nam / Nguyễn Ngọc Mão (ch.b.), Lê Trung Dũng, 
Nguyễn Thị Hồng Vân. - Tái bản. - H. : Khoa học xã hội. - 24cm
ĐTTS ghi: Viện Hàn lâm Khoa học xã hội Việt Nam. Viện Sử học
T. 15: Từ năm 1986 đến năm 2000. - 2017. - 448tr.
1. Lịc


---
## Phan 3: Chunking
2 chien luoc: **Co dinh** (500 ky tu) va **Theo cau truc** (Chuong/Phan/Muc)

In [7]:
def cat_co_dinh(text, do_dai=500, chong_lap=100):
    ket_qua, vi_tri, dem = [], 0, 0
    while vi_tri < len(text):
        doan = text[vi_tri:vi_tri + do_dai].strip()
        if doan:
            ket_qua.append({"id": f"f_{dem}", "text": doan, "loai": "co_dinh"})
            dem += 1
        vi_tri += do_dai - chong_lap
    return ket_qua


def cat_theo_cau_truc(text):
    mau = re.compile(
        r'((?:Chương|Phần|Mục|CHƯƠNG|PHẦN|MỤC|Bài|Chương|chương)\s+[IVXLCDM\d]+[.:\-]?\s*[^\n]*)',
        re.UNICODE | re.IGNORECASE
    )
    vi_tri_tim = list(mau.finditer(text))

    if len(vi_tri_tim) < 3:
        return cat_co_dinh(text, do_dai=800, chong_lap=150)

    ket_qua = []
    for idx, match in enumerate(vi_tri_tim):
        dau = match.start()
        cuoi = vi_tri_tim[idx + 1].start() if idx + 1 < len(vi_tri_tim) else len(text)
        doan = text[dau:cuoi].strip()
        if doan:
            ket_qua.append({
                "id": f"s_{idx}", "text": doan, "loai": "cau_truc",
                "tieu_de": match.group().strip()[:100],
            })
    return ket_qua


def thuc_hien_chunking(noi_dung, force=False):
    f1 = CFG["chunk_dir"] / "chunks_co_dinh.json"
    f2 = CFG["chunk_dir"] / "chunks_cau_truc.json"

    if f1.exists() and f2.exists() and not force:
        print("Da co chunks.")
        c1 = json.loads(f1.read_text(encoding="utf-8"))
        c2 = json.loads(f2.read_text(encoding="utf-8"))
        print(f"  Co dinh: {len(c1)} | Cau truc: {len(c2)}")
        return c1, c2

    c1 = cat_co_dinh(noi_dung)
    c2 = cat_theo_cau_truc(noi_dung)

    f1.write_text(json.dumps(c1, ensure_ascii=False, indent=2), encoding="utf-8")
    f2.write_text(json.dumps(c2, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Chunking xong: Co dinh={len(c1)} | Cau truc={len(c2)}")
    return c1, c2

In [8]:
if van_ban:
    chunks_cd, chunks_ct = thuc_hien_chunking(van_ban)
    print(f"\nVi du chunk co dinh:\n{chunks_cd[0]['text'][:200]}...")
    print(f"\nVi du chunk cau truc:\n{chunks_ct[0]['text'][:200]}...")

Da co chunks.
  Co dinh: 1825 | Cau truc: 16

Vi du chunk co dinh:
--- Trang 1/451 ---
VIỆN H À N  LÂM  K H O A  H ỌC  XẢ H ỘI VIỆT N A M  
VIỆN SỬ HỌC
NGUYỄN NGỌC M ÃO
(Chù biên)
LỊCH SỬ 
VIỆT NAM
TẬP 15 TỪ N Ă M  1986 Đ ẾN  N Ă M  2000

--- Trang 2/451 ---


--- Tr...

Vi du chunk cau truc:
Phan Văn Khải dự cuộc gặp không chính thức Thủ tướng 
ba nirórc Việt N am  - I ào và C am purhia tháng 10-1999. Thù tướng...


---
## Phan 4: Embedding
Model: **paraphrase-multilingual-MiniLM-L12-v2** (384 chieu, ho tro tieng Viet)

In [9]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

_bo_nhung = None

def tai_mo_hinh():
    global _bo_nhung
    if _bo_nhung is None:
        print(f"Dang tai model {CFG['embed_model']}...")
        _bo_nhung = SentenceTransformer(CFG["embed_model"])
        print(f"Model san sang (dim={_bo_nhung.get_sentence_embedding_dimension()})")
    return _bo_nhung

def tao_embedding(ds_text, batch_size=64):
    model = tai_mo_hinh()
    ds_vec = []
    for i in tqdm(range(0, len(ds_text), batch_size), desc="Embedding"):
        lo = ds_text[i:i + batch_size]
        ds_vec.append(model.encode(lo, show_progress_bar=False))
    return np.vstack(ds_vec).astype("float32")

def nhung_nhanh(texts):
    model = tai_mo_hinh()
    return model.encode(texts, show_progress_bar=False).astype("float32")

def chay_embedding(ds_chunks, ten_file, force=False):
    npy_path = CFG["emb_dir"] / f"{ten_file}.npy"
    if npy_path.exists() and not force:
        print(f"Da co: {npy_path}")
        return np.load(npy_path)
    ds_text = [c["text"] for c in ds_chunks]
    vectors = tao_embedding(ds_text)
    np.save(npy_path, vectors)
    print(f"Luu: {npy_path} | Shape: {vectors.shape}")
    return vectors

In [10]:
if van_ban:
    vec_cd = chay_embedding(chunks_cd, "vec_co_dinh")
    vec_ct = chay_embedding(chunks_ct, "vec_cau_truc")

Da co: data/embeddings/vec_co_dinh.npy
Da co: data/embeddings/vec_cau_truc.npy


---
## Phan 5: Vector Database
**FAISS** (nhanh) + **Chroma** (metadata)

In [11]:
# ===== FAISS =====
def tao_faiss(chunks, embeddings):
    import faiss
    CFG["vs_faiss"].mkdir(parents=True, exist_ok=True)
    idx = faiss.IndexFlatL2(embeddings.shape[1])
    idx.add(embeddings)
    faiss.write_index(idx, str(CFG["vs_faiss"] / "index.faiss"))
    with open(CFG["vs_faiss"] / "chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)
    print(f"FAISS: {idx.ntotal:,} vectors")
    return idx, chunks

def nap_faiss():
    import faiss
    idx = faiss.read_index(str(CFG["vs_faiss"] / "index.faiss"))
    with open(CFG["vs_faiss"] / "chunks.pkl", "rb") as f:
        chunks = pickle.load(f)
    return idx, chunks

def truy_van_faiss(q_vec, index, chunks, top_k=5):
    q = q_vec.reshape(1, -1).astype("float32")
    D, I = index.search(q, top_k)
    return [{"text": chunks[i]["text"], "metadata": chunks[i], "score": float(D[0][j])}
            for j, i in enumerate(I[0]) if i < len(chunks)]

# ===== CHROMA =====
def tao_chroma(chunks, embeddings):
    import chromadb
    CFG["vs_chroma"].mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(CFG["vs_chroma"]))
    try: client.delete_collection(CFG["collection"])
    except: pass
    col = client.create_collection(name=CFG["collection"], metadata={"hnsw:space": "cosine"})
    bs = 500
    for i in range(0, len(chunks), bs):
        lc = chunks[i:i+bs]
        le = embeddings[i:i+bs]
        col.add(
            ids=[f"{c['id']}_{i+j}" for j, c in enumerate(lc)],
            embeddings=le.tolist(),
            documents=[c["text"] for c in lc],
            metadatas=[{k: v for k, v in c.items() if k != "text" and v is not None} for c in lc],
        )
    print(f"Chroma: {col.count():,} vectors")
    return col

def nap_chroma():
    import chromadb
    client = chromadb.PersistentClient(path=str(CFG["vs_chroma"]))
    return client.get_collection(CFG["collection"])

def truy_van_chroma(q_vec, collection, top_k=5):
    res = collection.query(query_embeddings=[q_vec.tolist()], n_results=top_k)
    return [{"text": res["documents"][0][i], "metadata": res["metadatas"][0][i],
             "score": res["distances"][0][i]} for i in range(len(res["documents"][0]))]

In [12]:
if van_ban:
    faiss_idx, faiss_data = tao_faiss(chunks_ct, vec_ct)
    chroma_col = tao_chroma(chunks_ct, vec_ct)

FAISS: 16 vectors
Chroma: 16 vectors


---
## Phan 6: Benchmark
Sinh test set bang Groq, do **Recall@K**, **MRR**, **Latency**.

In [13]:
def sinh_bo_test(noi_dung_pdf, so_cau=30, force=False):
    file_test = CFG["out_dir"] / "benchmark" / "test_set.json"
    if file_test.exists() and not force:
        print(f"Da co: {file_test}")
        return json.loads(file_test.read_text(encoding="utf-8"))

    yeu_cau = (
        f"Dua vao tai lieu lich su Viet Nam duoi day, tao {so_cau} cau hoi.\n"
        "Moi cau hoi phai tra loi duoc tu noi dung tai lieu.\n"
        "Tra ve JSON array voi: \"question\" va \"section\" (ten muc lien quan).\n\n"
        f"Tai lieu:\n{noi_dung_pdf[:6000]}\n\n"
        "Chi tra ve JSON array, khong giai thich."
    )

    print(f"Dang sinh {so_cau} cau hoi bang Groq...")
    raw = hoi_llm(yeu_cau, max_tokens=4096)
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    bo_test = json.loads(raw.strip())

    file_test.parent.mkdir(parents=True, exist_ok=True)
    file_test.write_text(json.dumps(bo_test, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Da tao {len(bo_test)} cau hoi")
    return bo_test


def do_hieu_suat(ten, ham_tim, ds_q_vec, bo_test, top_k=5):
    r3, r5, mrr_vals, lat = [], [], [], []
    for i, item in enumerate(bo_test):
        t0 = time.perf_counter()
        kq = ham_tim(ds_q_vec[i], top_k=top_k)
        lat.append((time.perf_counter() - t0) * 1000)
        gt = item.get("section", "")
        if gt:
            f3 = any(gt.lower() in r.get("text", "").lower() for r in kq[:3])
            f5 = any(gt.lower() in r.get("text", "").lower() for r in kq[:5])
            r3.append(int(f3)); r5.append(int(f5))
            rank = next((j+1 for j,r in enumerate(kq) if gt.lower() in r.get("text","").lower()), 0)
            mrr_vals.append(1/rank if rank else 0)
    return {"Store": ten, "Recall@3": round(np.mean(r3),3) if r3 else 0,
            "Recall@5": round(np.mean(r5),3) if r5 else 0,
            "MRR": round(np.mean(mrr_vals),3) if mrr_vals else 0,
            "Latency(ms)": round(np.mean(lat),1)}


def chay_benchmark(van_ban_goc, force=False):
    file_kq = CFG["out_dir"] / "benchmark" / "ket_qua.csv"
    if file_kq.exists() and not force:
        print(f"Da co: {file_kq}")
        return pd.read_csv(file_kq)

    bo_test = sinh_bo_test(van_ban_goc)
    ds_cau_hoi = [item["question"] for item in bo_test]
    ds_q_vec = nhung_nhanh(ds_cau_hoi)

    def tim_f(qv, top_k=5): return truy_van_faiss(qv, faiss_idx, faiss_data, top_k)
    def tim_c(qv, top_k=5): return truy_van_chroma(qv, chroma_col, top_k)

    bang = []
    for ten, fn in [("FAISS", tim_f), ("Chroma", tim_c)]:
        print(f"  Benchmark {ten}...", end=" ")
        row = do_hieu_suat(ten, fn, ds_q_vec, bo_test)
        bang.append(row)
        print(f"R@3={row['Recall@3']:.3f} MRR={row['MRR']:.3f} Lat={row['Latency(ms)']}ms")

    # So sanh chunking
    import chromadb
    tmp_cl = chromadb.PersistentClient(path=str(CFG["vs_chroma"]))
    try: tmp_cl.delete_collection("tmp_cd")
    except: pass
    tmp_col = tmp_cl.create_collection(name="tmp_cd", metadata={"hnsw:space": "cosine"})
    bs = 500
    for i in range(0, len(chunks_cd), bs):
        lc = chunks_cd[i:i+bs]; le = vec_cd[i:i+bs]
        tmp_col.add(ids=[f"t_{i+j}" for j in range(len(lc))],
                    embeddings=le.tolist(), documents=[c["text"] for c in lc],
                    metadatas=[{k:v for k,v in c.items() if k!="text" and v is not None} for c in lc])
    def tim_cd(qv, top_k=5): return truy_van_chroma(qv, tmp_col, top_k)

    bang2 = []
    for ten, fn in [("Co dinh 500", tim_cd), ("Theo cau truc", tim_c)]:
        row = do_hieu_suat(ten, fn, ds_q_vec, bo_test)
        bang2.append(row)
    try: tmp_cl.delete_collection("tmp_cd")
    except: pass

    df1 = pd.DataFrame(bang)
    df2 = pd.DataFrame(bang2)
    df1.to_csv(file_kq, index=False)
    df2.to_csv(CFG["out_dir"] / "benchmark" / "chunking_compare.csv", index=False)

    print(f"\n--- VECTOR DB ---\n{df1.to_string(index=False)}")
    print(f"\n--- CHUNKING ---\n{df2.to_string(index=False)}")
    return df1

In [14]:
if van_ban:
    df_benchmark = chay_benchmark(van_ban)

Da co: outputs/benchmark/ket_qua.csv


---
## Phan 7: Hoi dap (RAG Query)
**Text** hoac **Anh chup** (OCR) → RAG → Groq → Tra loi

In [17]:
def hoi_dap_rag(cau_hoi, top_k=2):
    q_vec = nhung_nhanh([cau_hoi])[0]
    ket_qua = truy_van_chroma(q_vec, chroma_col, top_k=top_k)
    ngu_canh = "\n\n---\n\n".join(r["text"] for r in ket_qua)
    nguon = [{"tieu_de": r["metadata"].get("tieu_de", ""),
              "diem": round(float(r["score"]), 4),
              "tom_tat": r["text"][:200]} for r in ket_qua]

    prompt = (
        "Ban la chuyen gia lich su Viet Nam.\n"
        "Tra loi dua vao ngu canh, bang tieng Viet. Neu thieu thong tin, noi ro.\n\n"
        f"NGU CANH:\n{ngu_canh}\n\nCAU HOI: {cau_hoi}\n\nTRA LOI:"
    )
    tra_loi = hoi_llm(prompt)
    return {"cau_tra_loi": tra_loi, "nguon": nguon}


def hoi_dap_tu_anh(duong_dan_anh):
    import pytesseract
    from PIL import Image
    img = Image.open(duong_dan_anh)
    cau_hoi = pytesseract.image_to_string(img, lang="vie").strip()
    if not cau_hoi:
        return {"loi": "Khong doc duoc text tu anh"}
    print(f"OCR: {cau_hoi}")
    kq = hoi_dap_rag(cau_hoi)
    kq["cau_hoi_ocr"] = cau_hoi
    return kq

In [19]:
if van_ban:
    thu = hoi_dap_rag("Doi moi 1986 la gi?")
    print(f"Tra loi:\n{thu['cau_tra_loi']}")


Tra loi:
Đổi mới 1986 là một cuộc cải cách kinh tế và chính trị quan trọng ở Việt Nam, được thực hiện bởi Đảng Cộng sản Việt Nam. Cuộc đổi mới này được bắt đầu từ Đại hội Đảng lần thứ VI vào năm 1986, với mục tiêu chuyển đổi nền kinh tế từ một nền kinh tế tập trung, quan liêu, bao cấp sang một nền kinh tế thị trường định hướng xã hội chủ nghĩa.

Cuộc đổi mới 1986 đã đưa ra một số biện pháp quan trọng, bao gồm:

* Chuyển đổi cơ chế kinh tế từ tập trung, quan liêu, bao cấp sang cơ chế thị trường
* Tạo điều kiện cho sự phát triển của khu vực tư nhân và khu vực ngoài quốc doanh
* Đổi mới cơ chế quản lý và điều hành kinh tế
* Tăng cường vai trò của thị trường và cạnh tranh trong nền kinh tế
* Đổi mới hệ thống tài chính và ngân hàng

Cuộc đổi mới 1986 đã mang lại nhiều kết quả tích cực cho nền kinh tế Việt Nam, bao gồm:

* Tăng trưởng kinh tế nhanh chóng
* Tăng cường cạnh tranh và sáng tạo trong nền kinh tế
* Cải thiện đời sống của người dân
* Tăng cường vị thế của Việt Nam trên trường quốc 

---
## Phan 8: Agent thong minh (LangGraph)
- **rag**: Lich su VN → vector DB
- **web**: Ngoai pham vi → DuckDuckGo
- **calc**: Tinh toan → Calculator
- **direct**: Chung → Groq tra loi

In [34]:
def phan_loai(cau_hoi):
    q = cau_hoi.lower()
    tu_su = ["lich su", "trieu dai", "vua", "chua", "chien tranh", "tran danh",
             "khoi nghia", "doc lap", "thoi ky", "the ky", "nam nao", "su kien",
             "phong kien", "thuc dan", "cach mang", "bac ho", "ho chi minh",
             "dien bien phu", "nha tran", "nha ly", "nha le", "nha nguyen",
             "van lang", "au lac", "doi moi", "tran hung dao"]
    tu_tinh = ["tinh", "bao nhieu nam", "cong", "tru", "tu nam", "den nam"]

    if any(kw in q for kw in tu_tinh) and any(c.isdigit() for c in q):
        return "calc"
    if any(kw in q for kw in tu_su):
        return "rag"
    return "web"


def tao_react_agent():
    from langchain_groq import ChatGroq
    from langgraph.prebuilt import create_react_agent
    from langchain_core.tools import Tool
    from langchain_community.tools import DuckDuckGoSearchRun

    llm = ChatGroq(model=CFG["llm_model"], api_key=os.getenv("GROQ_API_KEY"), temperature=0.1)

    def _tim_su(query: str) -> str:
        qv = nhung_nhanh([query])[0]
        kq = truy_van_chroma(qv, chroma_col, top_k=1)
        if not kq: return "Khong tim thay."
        return "\n---\n".join(r["text"][:300] for r in kq)

    def _may_tinh(expr: str) -> str:
        try: return str(eval(expr, {"__builtins__": {}}))
        except Exception as e: return f"Loi: {e}"

    tools = [
        Tool(name="tim_lich_su", func=_tim_su,
             description="Tim kiem thong tin lich su Viet Nam. Input: cau hoi."),
        DuckDuckGoSearchRun(),
        Tool(name="may_tinh", func=_may_tinh,
             description="Tinh toan so hoc. Input: bieu thuc Python."),
    ]
    return create_react_agent(llm, tools,
        prompt="Ban la chuyen gia lich su Viet Nam. Tra loi tieng Viet. Dung tim_lich_su de tra cuu.")


def tao_langgraph_workflow():
    from langgraph.graph import StateGraph, END
    from typing import TypedDict
    from langchain_groq import ChatGroq

    llm = ChatGroq(model=CFG["llm_model"], api_key=os.getenv("GROQ_API_KEY"), temperature=0.1)

    class TT(TypedDict):
        cau_hoi: str
        tuyen: str
        ngu_canh: str
        tra_loi: str

    def node_router(s: TT) -> TT:
        return {**s, "tuyen": phan_loai(s["cau_hoi"])}

    def node_rag(s: TT) -> TT:
        qv = nhung_nhanh([s["cau_hoi"]])[0]
        kq = truy_van_chroma(qv, chroma_col, top_k=1)
        ctx = "\n---\n".join(r["text"] for r in kq)
        ans = llm.invoke(f"Dua vao ngu canh, tra loi tieng Viet:\n{ctx}\nCAU HOI: {s['cau_hoi']}\nTRA LOI:").content
        return {**s, "ngu_canh": ctx, "tra_loi": ans}

    def node_web(s: TT) -> TT:
        try:
            from langchain_community.tools import DuckDuckGoSearchRun
            web = DuckDuckGoSearchRun().run(s["cau_hoi"])
        except: web = "(Khong tim duoc)"
        ans = llm.invoke(f"Dua vao ket qua web, tra loi tieng Viet:\n{web}\nCAU HOI: {s['cau_hoi']}\nTRA LOI:").content
        return {**s, "ngu_canh": web, "tra_loi": ans}

    def node_calc(s: TT) -> TT:
        ans = llm.invoke(f"Tinh toan va tra loi tieng Viet:\nCAU HOI: {s['cau_hoi']}\nTRA LOI:").content
        return {**s, "ngu_canh": "", "tra_loi": ans}

    g = StateGraph(TT)
    g.add_node("router", node_router)
    g.add_node("rag", node_rag)
    g.add_node("web", node_web)
    g.add_node("calc", node_calc)
    g.set_entry_point("router")
    g.add_conditional_edges("router", lambda s: s["tuyen"],
                            {"rag": "rag", "web": "web", "calc": "calc"})
    for n in ["rag", "web", "calc"]: g.add_edge(n, END)
    return g.compile()

In [35]:
if van_ban:
    print("Khoi tao ReAct Agent...")
    react_agent = tao_react_agent()
    print("Khoi tao LangGraph Workflow...")
    graph_app = tao_langgraph_workflow()

    demo = "Doi moi 1986 dien ra nhu the nao?"
    print(f"\nCau hoi: {demo}")
    print(f"Phan loai: {phan_loai(demo)}")
    try:
        kq = graph_app.invoke({"cau_hoi": demo, "tuyen": "", "ngu_canh": "", "tra_loi": ""})
        print(f"Tra loi: {kq['tra_loi'][:400]}...")
    except Exception as e:
        print(f"Loi: {e}")

Khoi tao ReAct Agent...
Khoi tao LangGraph Workflow...

Cau hoi: Doi moi 1986 dien ra nhu the nao?
Phan loai: rag


/var/folders/gr/fmhgvmrd75x43lwhmh98pl_40000gn/T/ipykernel_8391/1318051430.py:42: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(llm, tools,


Tra loi: Đổi mới 1986 diễn ra như sau:

Đổi mới 1986 là một sự kiện quan trọng trong lịch sử Việt Nam, đánh dấu một bước ngoặt lớn trong quá trình phát triển của đất nước. Vào năm 1986, Đảng Cộng sản Việt Nam đã tổ chức Đại hội VI, tại đó đã thông qua Nghị quyết về đổi mới kinh tế và xã hội.

Đổi mới 1986 có mục tiêu chính là chuyển đổi nền kinh tế từ mô hình tập trung quan liêu bao cấp sang mô hình kinh t...


---
## Phan 9: Fine-tuning (LoRA)
Tao dataset Q&A bang Groq, xuat JSONL de train tren Colab.

In [1]:
def tao_dataset_qa(ds_chunks, so_qa=1, gioi_han=100, force=False):
    file_qa = CFG["dataset_dir"] / "qa_pairs.jsonl"
    if file_qa.exists() and not force:
        print(f"Da co: {file_qa}")
        return [json.loads(l) for l in file_qa.read_text(encoding="utf-8").splitlines() if l]

    chon = ds_chunks[:gioi_han]
    tat_ca = []
    print(f"Sinh Q&A tu {len(chon)} chunks...")

    for i, chunk in enumerate(chon):
        yc = (
            f"Dua vao doan van lich su sau, tao {so_qa} cap cau hoi - tra loi\n"
            f"bang tieng Viet, ngan gon.\n\n"
            f"DOAN VAN:\n{chunk['text'][:1500]}\n\n"
            'Tra ve JSON array: [{"question": "...", "answer": "..."}]\n'
            "Chi JSON, khong giai thich."
        )
        try:
            raw = hoi_llm(yc)
            if "```" in raw:
                raw = raw.split("```")[1]
                if raw.startswith("json"): raw = raw[4:]
            for p in json.loads(raw.strip()):
                tat_ca.append({"instruction": p["question"], "input": "", "output": p["answer"]})
        except Exception as e:
            print(f"  Chunk {i} loi: {e}")
        if (i+1) % 20 == 0:
            print(f"  [{i+1}/{len(chon)}] {len(tat_ca)} Q&A")


In [40]:
if van_ban:
    bo_qa = tao_dataset_qa(chunks_ct, so_qa=20, gioi_han=50)

Sinh Q&A tu 16 chunks...
Da tao 313 Q&A -> data/dataset/qa_pairs.jsonl


---
## Phan 10: Evaluation
- **Retrieval**: Recall@K, MRR, NDCG
- **Generation**: Faithfulness, Relevance (LLM-as-judge)

In [38]:
def danh_gia_sinh(ham_tim, ds_qv, bo_test, ten, so_mau=10):
    diem_tt, diem_lq = [], []
    for i, item in enumerate(bo_test[:so_mau]):
        cau_hoi = item["question"]
        kq = ham_tim(ds_qv[i], top_k=3)
        ctx = "\n---\n".join(r["text"][:300] for r in kq)
        tra_loi = hoi_llm(f"Tra loi dua vao ngu canh:\n{ctx}\nCAU HOI: {cau_hoi}\nTRA LOI:")
        raw = hoi_llm(
            f"Danh gia cau tra loi RAG:\nCAU HOI: {cau_hoi}\nNGU CANH: {ctx[:500]}\n"
            f"TRA LOI: {tra_loi[:300]}\n\n"
            'Cham diem 0.0-1.0: {{"faithfulness": X, "relevance": X}}\nChi tra ve JSON.'
        )
        try:
            if "```" in raw:
                raw = raw.split("```")[1]
                if raw.startswith("json"): raw = raw[4:]
            sc = json.loads(raw.strip())
            diem_tt.append(float(sc.get("faithfulness", 0)))
            diem_lq.append(float(sc.get("relevance", 0)))
        except:
            diem_tt.append(0); diem_lq.append(0)
    return {"Store": ten, "Faithfulness": round(np.mean(diem_tt),3),
            "Relevance": round(np.mean(diem_lq),3), "Samples": so_mau}


def ndcg_tai_k(kq, section_gt, k):
    dcg = sum(1/math.log2(i+2) for i, r in enumerate(kq[:k])
              if section_gt and section_gt.lower() in r.get("text","").lower())
    return dcg / (1/math.log2(2)) if dcg > 0 else 0


def chay_danh_gia(van_ban_goc, force=False):
    file_bc = CFG["out_dir"] / "evaluation" / "bao_cao.csv"
    if file_bc.exists() and not force:
        print(f"Da co: {file_bc}")
        return pd.read_csv(file_bc)

    bo_test = sinh_bo_test(van_ban_goc)
    ds_q = [it["question"] for it in bo_test]
    ds_qv = nhung_nhanh(ds_q)

    def tim_c(qv, top_k=5): return truy_van_chroma(qv, chroma_col, top_k)
    def tim_f(qv, top_k=5): return truy_van_faiss(qv, faiss_idx, faiss_data, top_k)

    print("Danh gia retrieval...")
    bang_r = []
    for ten, fn in [("FAISS", tim_f), ("Chroma", tim_c)]:
        r3, r5, mrr_l, ndcg_l, lat = [], [], [], [], []
        for i, item in enumerate(bo_test):
            gt = item.get("section", "")
            t0 = time.perf_counter()
            kq = fn(ds_qv[i], top_k=5)
            lat.append((time.perf_counter()-t0)*1000)
            if gt:
                r3.append(int(any(gt.lower() in r["text"].lower() for r in kq[:3])))
                r5.append(int(any(gt.lower() in r["text"].lower() for r in kq[:5])))
                rank = next((j+1 for j,r in enumerate(kq) if gt.lower() in r["text"].lower()), 0)
                mrr_l.append(1/rank if rank else 0)
                ndcg_l.append(ndcg_tai_k(kq, gt, 3))
        bang_r.append({"Store": ten, "Recall@3": round(np.mean(r3),3) if r3 else 0,
                       "Recall@5": round(np.mean(r5),3) if r5 else 0,
                       "MRR": round(np.mean(mrr_l),3) if mrr_l else 0,
                       "NDCG@3": round(np.mean(ndcg_l),3) if ndcg_l else 0,
                       "Latency(ms)": round(np.mean(lat),1)})
        print(f"  {ten}: R@3={bang_r[-1]['Recall@3']} MRR={bang_r[-1]['MRR']}")
    df_r = pd.DataFrame(bang_r)

    print("Danh gia generation (LLM-as-judge)...")
    bang_g = []
    for ten, fn in [("FAISS", tim_f), ("Chroma", tim_c)]:
        print(f"  {ten}...", end=" ")
        row = danh_gia_sinh(fn, ds_qv, bo_test, ten, so_mau=10)
        bang_g.append(row)
        print(f"Faith={row['Faithfulness']} Rel={row['Relevance']}")
    df_g = pd.DataFrame(bang_g)

    eval_dir = CFG["out_dir"] / "evaluation"
    eval_dir.mkdir(parents=True, exist_ok=True)
    df_r.to_csv(file_bc, index=False)
    df_r.to_csv(eval_dir / "retrieval.csv", index=False)
    df_g.to_csv(eval_dir / "generation.csv", index=False)

    try:
        import matplotlib.pyplot as plt
        import matplotlib; matplotlib.use("Agg")
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle("Danh gia RAG - Lich Su Viet Nam")
        cols_r = [c for c in ["Recall@3","Recall@5","MRR","NDCG@3"] if c in df_r.columns]
        df_r.set_index("Store")[cols_r].plot(kind="bar", ax=axes[0], rot=0)
        axes[0].set_title("Retrieval"); axes[0].set_ylim(0, 1.05)
        if not df_g.empty:
            cols_g = [c for c in ["Faithfulness","Relevance"] if c in df_g.columns]
            df_g.set_index("Store")[cols_g].plot(kind="bar", ax=axes[1], rot=0)
            axes[1].set_title("Generation"); axes[1].set_ylim(0, 1.05)
        plt.tight_layout()
        plt.savefig(eval_dir / "bieu_do.png", dpi=120); plt.close()
        print(f"Bieu do: {eval_dir / 'bieu_do.png'}")
    except: pass

    print(f"\n--- RETRIEVAL ---\n{df_r.to_string(index=False)}")
    print(f"\n--- GENERATION ---\n{df_g.to_string(index=False)}")
    return df_r

In [39]:
if van_ban:
    df_eval = chay_danh_gia(van_ban)

Da co: outputs/benchmark/test_set.json
Danh gia retrieval...
  FAISS: R@3=0.606 MRR=0.567
  Chroma: R@3=0.606 MRR=0.567
Danh gia generation (LLM-as-judge)...
  FAISS... Faith=0.78 Rel=0.57
  Chroma... Faith=0.56 Rel=0.38
Bieu do: outputs/evaluation/bieu_do.png

--- RETRIEVAL ---
 Store  Recall@3  Recall@5   MRR  NDCG@3  Latency(ms)
 FAISS     0.606     0.636 0.567   1.102          0.1
Chroma     0.606     0.636 0.567   1.106          0.7

--- GENERATION ---
 Store  Faithfulness  Relevance  Samples
 FAISS          0.78       0.57       10
Chroma          0.56       0.38       10


In [2]:
print("  1. Mo terminal")
print("  2. Chay: uvicorn api:app --reload --port 8000")
print("  3. Mo chat_ui.html trong trinh duyet")
print("  4. Hoac: http://localhost:8000/docs")

  1. Mo terminal
  2. Chay: uvicorn api:app --reload --port 8000
  3. Mo chat_ui.html trong trinh duyet
  4. Hoac: http://localhost:8000/docs
